# Predict CelebA Hair Attributes

Use the lightweight reviewed-bank attribute model to predict `length` and `curl` on the extracted CelebA hairstyle asset bank, and store prediction confidences for downstream dataset enrichment.

In [1]:
import json
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

PROJECT_ROOT = find_project_root()
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

import torch
from PIL import Image

from app.ml.inference import load_hairstyle_attribute_checkpoint, predict_hairstyle_attributes
from app.ml.transforms import ResizeImage

PROJECT_ROOT

WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon')

In [2]:
ASSET_ROOT = BACKEND_ROOT / 'data' / 'processed' / 'celeba_hair_rich_assets'
METADATA_ROOT = ASSET_ROOT / 'metadata'
CHECKPOINT_PATH = BACKEND_ROOT / 'checkpoints' / 'reviewed_hairstyle_basic' / 'basic_attribute_model.pt'

OUTPUT_ROOT = ASSET_ROOT / 'predictions'
PREDICTIONS_JSONL = OUTPUT_ROOT / 'predicted_basic_attributes.jsonl'
PREDICTIONS_CSV = OUTPUT_ROOT / 'predicted_basic_attributes.csv'
SUMMARY_JSON = OUTPUT_ROOT / 'prediction_summary.json'

IMAGE_SIZE = 224
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LIGHTWEIGHT_MODE = True
MAX_RECORDS = 250 if LIGHTWEIGHT_MODE else None

CHECKPOINT_PATH

WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon/backend/checkpoints/reviewed_hairstyle_basic/basic_attribute_model.pt')

In [3]:
metadata_rows = []
for path in sorted(METADATA_ROOT.glob('*.json')):
    metadata_rows.append(json.loads(path.read_text(encoding='utf-8')))

if MAX_RECORDS is not None:
    metadata_rows = metadata_rows[:MAX_RECORDS]

model, inverse_vocab, core_fields, payload = load_hairstyle_attribute_checkpoint(CHECKPOINT_PATH, device=DEVICE)
transform = ResizeImage((IMAGE_SIZE, IMAGE_SIZE))

print('Device:', DEVICE)
print('Assets to predict:', len(metadata_rows))
print('Core fields:', core_fields)
metadata_rows[0] if metadata_rows else None

Device: cpu
Assets to predict: 250
Core fields: ('length', 'curl')


{'asset_id': 'celeba_hair_000001',
 'source_dataset': 'CelebA',
 'original_celeba_file': '000002.jpg',
 'raw_image_path': 'D:\\Projects\\Personal Projects\\Hairstyle Recommender Live Tryon\\backend\\data\\raw\\celeba\\img_align_celeba\\img_align_celeba\\000002.jpg',
 'raw_mask_path': 'D:\\Projects\\Personal Projects\\Hairstyle Recommender Live Tryon\\backend\\data\\datasets\\celeba_hair_rich\\segmentation_masks\\000002_hair_mask.png',
 'image_path': 'D:\\Projects\\Personal Projects\\Hairstyle Recommender Live Tryon\\backend\\data\\processed\\celeba_hair_rich_assets\\images\\celeba_hair_000001.png',
 'mask_path': 'D:\\Projects\\Personal Projects\\Hairstyle Recommender Live Tryon\\backend\\data\\processed\\celeba_hair_rich_assets\\masks\\celeba_hair_000001_mask.png',
 'partition': 'train',
 'gender_label': 'female',
 'quality_score': 1.0,
 'confidence_bucket': 'high',
 'quality_flags': ['usable_candidate'],
 'hair_mask_stats': {'positive_pixels': 10858,
  'coverage_ratio': 0.279817,
  'b

In [4]:
prediction_rows = []
for metadata in metadata_rows:
    with Image.open(metadata['image_path']).convert('RGB') as image:
        image_tensor = transform(image)

    predictions = predict_hairstyle_attributes(model, image_tensor, inverse_vocab)
    prediction_rows.append({
        'asset_id': metadata['asset_id'],
        'image_path': metadata['image_path'],
        'gender_label': metadata.get('gender_label'),
        'confidence_bucket': metadata.get('confidence_bucket'),
        'predictions': predictions,
    })

print('Predictions created:', len(prediction_rows))
prediction_rows[0] if prediction_rows else None

Predictions created: 250


{'asset_id': 'celeba_hair_000001',
 'image_path': 'D:\\Projects\\Personal Projects\\Hairstyle Recommender Live Tryon\\backend\\data\\processed\\celeba_hair_rich_assets\\images\\celeba_hair_000001.png',
 'gender_label': 'female',
 'confidence_bucket': 'high',
 'predictions': {'length': {'label': 'short',
   'confidence': 0.7883212566375732},
  'curl': {'label': 'straight', 'confidence': 0.698000431060791}}}

In [5]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

with PREDICTIONS_JSONL.open('w', encoding='utf-8') as handle:
    for row in prediction_rows:
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')

flat_rows = []
for row in prediction_rows:
    flat_rows.append({
        'asset_id': row['asset_id'],
        'image_path': row['image_path'],
        'gender_label': row['gender_label'],
        'confidence_bucket': row['confidence_bucket'],
        'pred_length': row['predictions']['length']['label'],
        'pred_length_confidence': row['predictions']['length']['confidence'],
        'pred_curl': row['predictions']['curl']['label'],
        'pred_curl_confidence': row['predictions']['curl']['confidence'],
    })

prediction_frame = pd.DataFrame(flat_rows)
prediction_frame.to_csv(PREDICTIONS_CSV, index=False, encoding='utf-8')

summary = {
    'total_assets': len(prediction_rows),
    'pred_length_counts': prediction_frame['pred_length'].value_counts().to_dict(),
    'pred_curl_counts': prediction_frame['pred_curl'].value_counts().to_dict(),
    'avg_length_confidence': round(float(prediction_frame['pred_length_confidence'].mean()), 4),
    'avg_curl_confidence': round(float(prediction_frame['pred_curl_confidence'].mean()), 4),
}
SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('Wrote:', PREDICTIONS_JSONL)
print('Wrote:', PREDICTIONS_CSV)
print('Wrote:', SUMMARY_JSON)

Wrote: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\processed\celeba_hair_rich_assets\predictions\predicted_basic_attributes.jsonl
Wrote: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\processed\celeba_hair_rich_assets\predictions\predicted_basic_attributes.csv
Wrote: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\processed\celeba_hair_rich_assets\predictions\prediction_summary.json


In [6]:
prediction_frame.head(20)

,asset_id,image_path,gender_label,confidence_bucket,pred_length,pred_length_confidence,pred_curl,pred_curl_confidence
0,celeba_hair_000001,D:\Projects\Personal Projects\Hairstyle Recomm...,female,high,short,0.788321,straight,0.698000
1,celeba_hair_000002,D:\Projects\Personal Projects\Hairstyle Recomm...,female,high,short,0.650244,straight,0.736030
2,celeba_hair_000003,D:\Projects\Personal Projects\Hairstyle Recomm...,male,medium,short,0.687539,straight,0.683670
3,celeba_hair_000004,D:\Projects\Personal Projects\Hairstyle Recomm...,male,medium,short,0.467601,straight,0.512496
4,celeba_hair_000005,D:\Projects\Personal Projects\Hairstyle Recomm...,female,high,short,0.475137,straight,0.491533
5,celeba_hair_000006,D:\Projects\Personal Projects\Hairstyle Recomm...,male,medium,short,0.548526,straight,0.502465
6,celeba_hair_000007,D:\Projects\Personal Projects\Hairstyle Recomm...,male,medium,short,0.811774,straight,0.738698
7,celeba_hair_000008,D:\Projects\Personal Projects\Hairstyle Recomm...,female,high,short,0.840726,straight,0.774655
8,celeba_hair_000009,D:\Projects\Personal Projects\Hairstyle Recomm...,male,medium,short,0.704176,straight,0.640301
9,celeba_hair_000010,D:\Projects\Personal Projects\Hairstyle Recomm...,male,medium,short,0.495767,wavy,0.548884


In [7]:
pd.Series(summary)

total_assets                                       250
pred_length_counts          {'short': 230, 'long': 20}
pred_curl_counts         {'straight': 203, 'wavy': 47}
avg_length_confidence                           0.6599
avg_curl_confidence                              0.637
dtype: object